In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")


In [0]:
import pandas as pd
from src.data.collectors.weather import _DNO_REGIONS

# Load the summary df from both models
results_04 = pd.read_csv("/Volumes/workspace/default/models/04_train_regional_models.csv")
results_04 = results_04[results_04['region_id'] <= 14]

results_08 = pd.read_csv("/Volumes/workspace/default/models/08_train_regional_models_with_weather.csv")


print(f"Model 04 (without weather): {len(results_04)} rows")
print(f"Model 08 (with weather): {len(results_08)} rows")

# Display both
print("\n=== Model without weather ===")
print(results_04.head())

print("\n=== Model with weather ===")
print(results_08.head())

  1. Filter both DataFrames to P50 (alpha == 0.5)
  2. Add the model_type column to each
  3. Combine with pd.concat
  4. Plot with matplotlib or seaborn


In [0]:
print(results_04.sort_values(by="region_id"))

In [0]:
print(results_08.sort_values(by="region_id"))

In [0]:
# Add model type to both
results_04['model_type'] = 'no_weather'
results_08['model_type'] = 'with_weather'


# Filter by alpha
results_04_50 = results_04[results_04['alpha'] == 0.5]
results_08_50 = results_08[results_08['alpha'] == 0.5]

# Combine them for comparison
alpha_50_combined = pd.concat([results_04_50, results_08_50])


%pip install plotly
import plotly.express as px

x = alpha_50_combined["region"]
y = alpha_50_combined["pinball_loss"]
color = alpha_50_combined["model_type"]
barmode = "group"

# Plot the results
fig = px.bar(x=x, y=y, color=color, barmode=barmode)
fig.show()



In [0]:
regions_df = pd.DataFrame([
      {"region_id": k, "region_name": v["name"], "latitude": v["latitude"], "longitude": v["longitude"]}
      for k, v in _DNO_REGIONS.items()
  ])

regions_df = regions_df.drop(columns=["region_name"])

# Join locations on to the WITH WEATHER results
results_08_50_combined = results_08_50.merge(regions_df, on="region_id")

# As I want to plot the pinball loss I am going to round it
results_08_50_combined["pinball_loss"] = results_08_50_combined["pinball_loss"].round(2)

# I want to bin the pinball loss into low medium and high in order to help the user make sense of it
# ❯ 1.2->2.1 = LOW, 2.1->5.5 = MEDIUM, 5.5->6.5 = HIGH 
bins = [1.2, 2.1, 5.5, 6.6]
labels=["HIGH", "MEDIUM", "LOW"]
results_08_50_combined["Accuracy"] = pd.cut(results_08_50_combined["pinball_loss"], bins=bins, labels=labels)
# DBTITLE 1,Plotting the results
#

In [0]:
fig = px.scatter_geo(
      results_08_50_combined,
      lat="latitude",
      lon="longitude",
      color="pinball_loss",
      size="pinball_loss",
      hover_name="region",
      hover_data={"latitude": False, "longitude": False, "Accuracy": True, "pinball_loss": True"},
      scope="europe",
      fitbounds="locations",
      title="P50 Pinball Loss by DNO Region"
  )
fig.show()

In [0]:
fig = px.scatter_mapbox(
    results_08_50_combined,
    lat="latitude",
    lon="longitude",
    color="pinball_loss",
    size="pinball_loss",
    hover_name="region",
    hover_data={"latitude": False, "longitude": False, "Accuracy": True, "pinball_loss": True},
    color_continuous_scale="Viridis",
    mapbox_style="open-street-map",
    zoom=4,
    center={"lat": 54.5, "lon": -2.5},
    title="P50 Pinball Loss by DNO Region"
)
fig.show()

In [0]:
results_08_50_combined[results_08_50_combined["region"] == "South West England"]